# 13. Figuras de la evaluación oficial

Este cuaderno lee únicamente las tablas consolidadas por 11. No entrena modelos ni ejecuta inferencia.

## A. Abrir las tablas verificadas

- **Qué hace:** carga métricas y matrices de la evaluación oficial.
- **Para qué:** representar exactamente los resultados 70, 76, 63 y 64.
- **Por qué:** las figuras deben proceder de la misma evidencia que las tablas de la tesis.

In [ ]:
%pip install -q pandas openpyxl matplotlib

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

ENTRADA = RAIZ / 'TABLAS/11_evaluacion_final'
SALIDA = RAIZ / 'TABLAS/13_figuras'
SALIDA.mkdir(parents=True, exist_ok=True)
METRICAS = ENTRADA / 'metricas_globales.xlsx'
CLASES = ENTRADA / 'metricas_por_clase.xlsx'
MATRICES = ENTRADA / 'matrices_confusion.xlsx'
MANIFIESTO = ENTRADA / 'manifest_evaluacion_oficial.json'
for ruta in [METRICAS, CLASES, MATRICES, MANIFIESTO]:
    if not ruta.is_file():
        raise FileNotFoundError(f'Ejecuta primero 11: falta {ruta.name}')
manifest = json.loads(MANIFIESTO.read_text(encoding='utf-8'))
if manifest.get('estado') != 'evaluacion_oficial_consolidada_sin_inferencia_nueva':
    raise ValueError('La evaluación de entrada no está identificada como oficial.')

metricas = pd.read_excel(METRICAS)
metricas_clase = pd.read_excel(CLASES)
if dict(zip(metricas.modelo, metricas.aciertos)) != {
    'M0_base_TASS': 70, 'M1_ajustado_SMPS': 76,
    'M2_ajustado_boliviano': 63, 'M3_SMPS_boliviano': 64,
}:
    raise ValueError('Las métricas no corresponden a la evaluación oficial.')
plt.rcParams['font.family'] = ['Liberation Serif', 'Times New Roman', 'serif']
plt.rcParams['font.size'] = 11
display(metricas.head(10))

## B. Matrices de confusión

- **Qué hace:** presenta los cuatro modelos en una cuadrícula 2×2.
- **Para qué:** mostrar aciertos y confusiones por polaridad.
- **Por qué:** la exactitud global no muestra qué clase origina cada error.

In [ ]:
MODELOS = metricas['modelo'].tolist()
NOMBRES = {'M0_base_TASS':'M0: Base TASS','M1_ajustado_SMPS':'M1: Ajustado SMPS',
           'M2_ajustado_boliviano':'M2: Ajustado boliviano','M3_SMPS_boliviano':'M3: SMPS + boliviano'}
ORDEN = ['negativa', 'neutral', 'positiva']
matrices = {m: pd.read_excel(MATRICES, sheet_name=f"{m.split('_')[0]}_cantidad", index_col=0).loc[ORDEN, ORDEN].to_numpy() for m in MODELOS}
fig, ejes = plt.subplots(2, 2, figsize=(11, 9))
maximo = max(x.max() for x in matrices.values())
for eje, nombre in zip(ejes.flat, MODELOS):
    matriz = matrices[nombre]; eje.imshow(matriz, cmap='Blues', vmin=0, vmax=maximo)
    eje.set_xticks(range(3), ORDEN); eje.set_yticks(range(3), ORDEN)
    eje.set_xlabel('Predicción'); eje.set_ylabel('Etiqueta humana'); eje.set_title(NOMBRES[nombre], fontweight='bold')
    for i in range(3):
        for j in range(3): eje.text(j, i, int(matriz[i,j]), ha='center', va='center', fontweight='bold')
fig.suptitle('Matrices de confusión en la prueba independiente', fontweight='bold')
fig.text(0.01, 0.01, 'Fuente: elaboración propia con las salidas verificadas del cuaderno 11.', fontsize=9)
fig.tight_layout(rect=[0,0.035,1,0.96]); fig.savefig(SALIDA/'matrices_confusion_modelos.png', dpi=180, bbox_inches='tight'); plt.show()

## C. Accuracy y F1 macro con intervalos

- **Qué hace:** compara las métricas globales y sus intervalos de confianza.
- **Para qué:** mostrar rendimiento e incertidumbre en una misma figura.
- **Por qué:** la prueba contiene 100 observaciones.

In [ ]:
x=np.arange(len(metricas)); ancho=0.36
fig,eje=plt.subplots(figsize=(10,5.8))
err_acc=np.vstack([metricas.accuracy-metricas.ic95_accuracy_inferior,metricas.ic95_accuracy_superior-metricas.accuracy])
err_f1=np.vstack([metricas.f1_macro-metricas.ic95_f1_inferior,metricas.ic95_f1_superior-metricas.f1_macro])
b1=eje.bar(x-ancho/2,metricas.accuracy,ancho,yerr=err_acc,capsize=4,label='Accuracy')
b2=eje.bar(x+ancho/2,metricas.f1_macro,ancho,yerr=err_f1,capsize=4,label='F1 macro')
eje.set_xticks(x,[NOMBRES[m] for m in MODELOS]); eje.set_ylim(0,1); eje.set_ylabel('Valor')
eje.set_title('Comparación oficial con intervalos de confianza',fontweight='bold'); eje.legend(); eje.grid(axis='y',alpha=.25)
for barras in [b1,b2]:
    for barra in barras: eje.text(barra.get_x()+barra.get_width()/2,barra.get_height()+.02,f'{barra.get_height():.3f}',ha='center')
fig.text(.01,.01,'Fuente: elaboración propia con las salidas verificadas del cuaderno 11.',fontsize=9)
fig.tight_layout(rect=[0,.035,1,1]); fig.savefig(SALIDA/'comparacion_accuracy_f1.png',dpi=180,bbox_inches='tight'); plt.show()

## D. F1 por clase

- **Qué hace:** compara F1 para negativa, neutral y positiva.
- **Para qué:** hacer visible el soporte reducido de la clase positiva.
- **Por qué:** el promedio global puede ocultar diferencias entre clases.

In [ ]:
tabla=metricas_clase.pivot(index='modelo',columns='clase',values='f1').reindex(MODELOS)
soportes=metricas_clase.groupby('clase').soporte.first().reindex(ORDEN)
fig,eje=plt.subplots(figsize=(10,5.8)); tabla[ORDEN].plot(kind='bar',ax=eje,color=['#4C78A8','#F58518','#54A24B'])
eje.set_xticklabels([NOMBRES[m] for m in MODELOS],rotation=0); eje.set_ylim(0,1); eje.set_xlabel('Modelo'); eje.set_ylabel('F1')
eje.set_title('F1 por clase en la prueba independiente',fontweight='bold')
eje.legend([f'{c} (n={int(soportes[c])})' for c in ORDEN],title='Clase y soporte'); eje.grid(axis='y',alpha=.25)
fig.text(.01,.01,'Fuente: elaboración propia con las salidas verificadas del cuaderno 11.',fontsize=9)
fig.tight_layout(rect=[0,.035,1,1]); fig.savefig(SALIDA/'f1_por_clase.png',dpi=180,bbox_inches='tight'); plt.show()
print('Figuras guardadas en: TABLAS/13_figuras')